# Medallion artifact check
This notebook verifies that the bronze and silver DuckDB artifacts exist, then previews the tables so you can inspect the actual data.

## What this checks
- whether the DuckDB database exists
- whether the bronze and silver schema/table are present
- a few sample rows from each table

In [ ]:
from pathlib import Path

import duckdb

DB_PATH = Path("/workspaces/financial-transactions-etl/data/db/transactions.duckdb")
print(f"DB exists: {DB_PATH.exists()}")
print(f"DB path: {DB_PATH}")

assert DB_PATH.exists(), f"Missing database at {DB_PATH}"

con = duckdb.connect(str(DB_PATH))
print("DuckDB connected")

schema_rows = con.execute(
    "SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema IN ('bronze','silver') ORDER BY table_schema, table_name"
).fetchall()
print("Medallion tables:")
for row in schema_rows:
    print(" -", row[0], row[1])

assert any(r[0] == "bronze" and r[1] == "transaction" for r in schema_rows), (
    "Missing bronze.transaction"
)
assert any(r[0] == "silver" and r[1] == "transaction" for r in schema_rows), (
    "Missing silver.transaction"
)

In [ ]:
bronze_count = con.execute('SELECT COUNT(*) FROM bronze."transaction"').fetchone()[0]
silver_count = con.execute('SELECT COUNT(*) FROM silver."transaction"').fetchone()[0]

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print("Bronze columns:", con.execute('DESCRIBE bronze."transaction"').fetchall())
print("Silver columns:", con.execute('DESCRIBE silver."transaction"').fetchall())

In [ ]:
bronze_df = con.execute('SELECT * FROM bronze."transaction" LIMIT 10').fetchdf()
bronze_df.head()

In [ ]:
silver_df = con.execute('SELECT * FROM silver."transaction" LIMIT 10').fetchdf()
silver_df.head()

In [ ]:
gold_tables = con.execute(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'gold'
    ORDER BY table_name
    """
).fetchall()

expected_tables = {
    "dim_clients",
    "dim_advisors",
    "dim_instruments",
    "dim_portfolios",
    "dim_transaction_types",
    "dim_client_portfolios",
    "dim_channels",
    "dim_source_systems",
    "fact_transactions",
}
existing_tables = {table_name for (table_name,) in gold_tables}
missing_tables = sorted(expected_tables - existing_tables)
assert not missing_tables, f"Missing gold tables: {missing_tables}"

print("Gold tables:")
for (table_name,) in gold_tables:
    row_count = con.execute(f'SELECT COUNT(*) FROM gold."{table_name}"').fetchone()[0]
    print(f"\n{table_name}: {row_count} rows")
    print(con.execute(f'DESCRIBE gold."{table_name}"').fetchdf().to_string(index=False))

print("\nFact transaction preview with dimension labels:")
fact_preview = con.execute(
    """
    SELECT
        f.transaction_id,
        c.client_name,
        a.advisor_name,
        i.instrument_name,
        p.portfolio_id,
        dtt.transaction_type_name,
        f.transaction_date,
        f.notes,
        f.is_flagged
    FROM gold.fact_transactions AS f
    LEFT JOIN gold.dim_clients AS c ON c.client_id = f.client_id
    LEFT JOIN gold.dim_advisors AS a ON a.advisor_id = f.advisor_id
    LEFT JOIN gold.dim_instruments AS i ON i.instrument_id = f.instrument_id
    LEFT JOIN gold.dim_portfolios AS p ON p.portfolio_id = f.portfolio_id
    LEFT JOIN gold.dim_transaction_types AS dtt ON dtt.transaction_type_id = f.transaction_type_id
    ORDER BY f.transaction_date NULLS LAST
    LIMIT 10
    """
).fetchdf()
display(fact_preview)

print("\nClient-to-portfolio snapshot:")
display(
    con.execute(
        "SELECT client_id, portfolio_id FROM gold.dim_client_portfolios LIMIT 10"
    ).fetchdf()
)

In [ ]:
con.close()